In [96]:
import os
import numpy
import time
import torch
import torch.nn.functional as F
import torchaudio
from torchaudio import transforms
import numpy as np
import bc_resnet_model
from IPython.display import Audio

# ================================
# 설정
# ================================
EPS         = 1e-9
SAMPLE_RATE = 16000
TARGET_LEN  = 16000  # 1초
SCALE       = 2
N_CLASS     = 2
CLASSES     = ['non-wakeword', 'wakeword']
THRESHOLD   = 0.7

# ================================
# 모델 로드
# ================================
device = torch.device('cpu')
model  = bc_resnet_model.BcResNetModel(
    n_class=N_CLASS,
    scale=SCALE,
    use_subspectral=True
)

import torch.nn as nn
model.head_conv = nn.Conv2d(32 * SCALE, N_CLASS, kernel_size=1)
model.load_state_dict(torch.load('./train_model/best_horiya2_11.pt', map_location='cpu'))
model.eval()
print("모델 로드 완료!")

# ================================
# 전처리 (re9ulus 방식)
# ================================
to_mel = transforms.MelSpectrogram(
    sample_rate=SAMPLE_RATE,
    n_fft=1024,
    f_max=8000,
    n_mels=40
)

def preprocess(waveform, sr):
    # 스테레오 → 모노
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)

    # 리샘플링
    if sr != SAMPLE_RATE:
        waveform = transforms.Resample(sr, SAMPLE_RATE)(waveform)

    # 패딩/자르기 (1초)
    length = waveform.shape[1]
    if length > TARGET_LEN:
        waveform = waveform[:, :TARGET_LEN]
    elif length < TARGET_LEN:
        waveform = F.pad(waveform, (0, TARGET_LEN - length))

    # 로그 멜 스펙트로그램
    log_mel = (to_mel(waveform) + EPS).log2()
    return log_mel.unsqueeze(0)  # (1, 1, 40, T)

def preprocess2(waveform, sr):
    # 스테레오 → 모노
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)

    # 리샘플링
    if sr != SAMPLE_RATE:
        waveform = transforms.Resample(sr, SAMPLE_RATE)(waveform)

    # 패딩/자르기 (1초)
    length = waveform.shape[1]
    if length > TARGET_LEN:
        waveform = waveform[:, :TARGET_LEN]
    elif length < TARGET_LEN:
        waveform = F.pad(waveform, (0, TARGET_LEN - length))

    return waveform

모델 로드 완료!


In [6]:
base_path = "/home/isol/work/bc_resnet_re9ulus/test_audio/1m/positive"
wav_path_list = [os.path.join(base_path, i) for i in os.listdir(base_path)]

waveform, sr = torchaudio.load(wav_path_list[0])
new_waveform = preprocess2(waveform, sr=sr)
print(len(new_waveform[0])/16000)
Audio(new_waveform, rate=sr)

1.0


In [88]:
16*30

480

In [89]:
# ================================
# 테스트
# ================================

base_path = "/home/isol/work/bc_resnet_re9ulus/test_audio/1m/positive"
wav_path_list = sorted([os.path.join(base_path, i) for i in os.listdir(base_path)])

inference_ms_list = []
probs_list = []

def inference_wav_file(wav_path, model):
    waveform, sr = torchaudio.load(wav_path)
    spec = preprocess(waveform, sr)

    with torch.no_grad():
        output = model(spec)
        probs = F.softmax(output.squeeze(), dim=0)


    print("="*35)
    print(f"파일 이름: {wav_path}")
    print(f"길이: {len(waveform)}")
    for i, cls in enumerate(CLASSES):
        bar    = "█" * int(probs[i].item() * 20)
        marker = " ← 감지!" if (cls == 'wakeword' and probs[i].item() >= THRESHOLD) else ""
        print(f"{cls:10s}: {probs[i].item()*100:5.1f}% {bar}{marker}")
    print("="*35)

    return probs

def realtime_inference_wav_file(wav_path, model):
    waveform, sr = torchaudio.load(wav_path)

    inference_result = []

    HOP_SIZE = int(SAMPLE_RATE * 0.03)

    if waveform.shape[1] > SAMPLE_RATE:
        for start in range(0, waveform.shape[1]-SAMPLE_RATE, HOP_SIZE):
            spec = preprocess(waveform[:,start:start+16000], sr)

            with torch.no_grad():
                output = model(spec)
                probs = F.softmax(output.squeeze(), dim=0)

            if probs[1] >= THRESHOLD:
                inference_result.append(1)
            else:
                inference_result.append(0)
    else:
        spec = preprocess(waveform, sr)

        with torch.no_grad():
            output = model(spec)
            probs = F.softmax(output.squeeze(), dim=0)

            if probs[1] >= THRESHOLD:
                inference_result.append(1)
            else:
                inference_result.append(0)

    return inference_result

for wav_path in wav_path_list:
    start = time.time()
    probs = inference_wav_file(wav_path, model)
    end = time.time()

    probs_list.append(probs)

    inference_ms = (end - start) * 1000
    inference_ms_list.append(inference_ms)

mean_inference_ms = np.mean(inference_ms_list)
print(f"평균 추론 시간: {mean_inference_ms:.2f}ms")
print(f"30ms 실시간 가능 여부: {'✅ 가능' if mean_inference_ms < 30 else '❌ 느림, 스텝 늘려야 함'}")

파일 이름: /home/isol/work/bc_resnet_re9ulus/test_audio/1m/positive/positive_00001.wav
길이: 1
non-wakeword:  70.3% ██████████████
wakeword  :  29.7% █████
파일 이름: /home/isol/work/bc_resnet_re9ulus/test_audio/1m/positive/positive_00002.wav
길이: 1
non-wakeword:  26.9% █████
wakeword  :  73.1% ██████████████ ← 감지!
파일 이름: /home/isol/work/bc_resnet_re9ulus/test_audio/1m/positive/positive_00003.wav
길이: 1
non-wakeword:  79.0% ███████████████
wakeword  :  21.0% ████
파일 이름: /home/isol/work/bc_resnet_re9ulus/test_audio/1m/positive/positive_00004.wav
길이: 1
non-wakeword:  84.3% ████████████████
wakeword  :  15.7% ███
파일 이름: /home/isol/work/bc_resnet_re9ulus/test_audio/1m/positive/positive_00005.wav
길이: 1
non-wakeword:  95.0% ██████████████████
wakeword  :   5.0% █
파일 이름: /home/isol/work/bc_resnet_re9ulus/test_audio/1m/positive/positive_00006.wav
길이: 1
non-wakeword:  40.2% ████████
wakeword  :  59.8% ███████████
파일 이름: /home/isol/work/bc_resnet_re9ulus/test_audio/1m/positive/positive_00007.wav
길이: 1
non-w

In [53]:
wav1_list = []
wav2_list = []
wav3_list = []

for i, probs in enumerate(probs_list):
    if probs[1] >= 0.7:
        if i < 33:
            wav1_list.append(1)
        elif i < 66:
            wav2_list.append(1)
        else:
            wav3_list.append(1)
    else:
        if i < 33:
            wav1_list.append(0)
        elif i < 66:
            wav2_list.append(0)
        else:
            wav3_list.append(0)

In [59]:
a = sum(wav1_list) / len(wav1_list)
b = sum(wav2_list) / len(wav2_list)
c = sum(wav3_list) / len(wav3_list)

(a + b + c) / 3

0.7183600713012478

In [ ]:
base_path = "/home/isol/work/bc_resnet_re9ulus/test_audio/1m/positive_extend"
wav_path_list = sorted([os.path.join(base_path, i) for i in os.listdir(base_path)])

total_result_list = []

for wav_path in wav_path_list:
    result_list = realtime_inference_wav_file(wav_path, model)

    print("="*35)
    print(f"파일 이름: {wav_path}")
    print(f"길이: {len(waveform)}")
    if sum(result_list) > 0:
        print("감지!")
        total_result_list.append(1)
    else:
        print("실패")
        total_result_list.append(0)
    print("="*35)

파일 이름: /home/isol/work/bc_resnet_re9ulus/test_audio/1m/positive_extend/positive_00001.wav
길이: 1
감지!
파일 이름: /home/isol/work/bc_resnet_re9ulus/test_audio/1m/positive_extend/positive_00002.wav
길이: 1
감지!
파일 이름: /home/isol/work/bc_resnet_re9ulus/test_audio/1m/positive_extend/positive_00003.wav
길이: 1
감지!
파일 이름: /home/isol/work/bc_resnet_re9ulus/test_audio/1m/positive_extend/positive_00004.wav
길이: 1
감지!
파일 이름: /home/isol/work/bc_resnet_re9ulus/test_audio/1m/positive_extend/positive_00005.wav
길이: 1
실패
파일 이름: /home/isol/work/bc_resnet_re9ulus/test_audio/1m/positive_extend/positive_00006.wav
길이: 1
감지!
파일 이름: /home/isol/work/bc_resnet_re9ulus/test_audio/1m/positive_extend/positive_00007.wav
길이: 1
감지!
파일 이름: /home/isol/work/bc_resnet_re9ulus/test_audio/1m/positive_extend/positive_00008.wav
길이: 1
감지!
파일 이름: /home/isol/work/bc_resnet_re9ulus/test_audio/1m/positive_extend/positive_00009.wav
길이: 1
감지!
파일 이름: /home/isol/work/bc_resnet_re9ulus/test_audio/1m/positive_extend/positive_00010.wav
길이: 1
감지!
파

In [97]:
np.mean(total_result_list)

np.float64(0.96)

In [98]:
total_result_list

[1,
 1,
 1,
 1,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
 1,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1]